# PostgreSQL queries — 3NF before vs after

Use this notebook to explore why 3NF matters: supplier attributes repeated on every
product cause update anomalies; after normalization, one supplier row is enough.

Uses the shared `scripts.db` helpers (same connection as `make seed` / `seed.ipynb`).

Run `make compose_up` and seed via `seed.ipynb` or `make seed`, then open this notebook at `http://localhost:8888`.

In [ ]:
from scripts.db import query

## 2NF table (violates 3NF)

Transitive dependency: `supplier_name` / `supplier_city` depend on `supplier_id`, not on `sku`.

In [ ]:
query(
    """
    SELECT sku, name, unit_price_cents, supplier_id, supplier_name, supplier_city
    FROM products_2nf
    ORDER BY supplier_id, sku
    LIMIT 15
    """
)

### Redundancy: same supplier, repeated name/city

Count how many products repeat each supplier's attributes.

In [ ]:
query(
    """
    SELECT
      supplier_id,
      supplier_name,
      supplier_city,
      COUNT(*) AS product_count
    FROM products_2nf
    GROUP BY supplier_id, supplier_name, supplier_city
    ORDER BY product_count DESC
    """
)

### Pain point: update anomaly

Changing a supplier's city requires updating **every** product that references that supplier.
The next cell shows how many rows would change for the most common supplier.

In [ ]:
query(
    """
    SELECT supplier_id, supplier_city, COUNT(*) AS rows_to_update
    FROM products_2nf
    WHERE supplier_id = (
      SELECT supplier_id
      FROM products_2nf
      GROUP BY supplier_id
      ORDER BY COUNT(*) DESC
      LIMIT 1
    )
    GROUP BY supplier_id, supplier_city
    """
)

## 3NF tables

Supplier attributes live once in `suppliers`; `products` keeps only `supplier_id`.

In [ ]:
query(
    """
    SELECT id, name, city
    FROM suppliers
    ORDER BY id
    """
)

In [ ]:
query(
    """
    SELECT sku, name, unit_price_cents, supplier_id
    FROM products
    ORDER BY supplier_id, sku
    LIMIT 15
    """
)

### Same question, cleaner answer: city update touches one row

After 3NF, changing a supplier city updates exactly one row in `suppliers`.

In [ ]:
query(
    """
    UPDATE suppliers
    SET city = city || ' (updated)'
    WHERE id = (SELECT id FROM suppliers ORDER BY id LIMIT 1)
    RETURNING id, name, city
    """
)

### Join for product + supplier details

In [ ]:
query(
    """
    SELECT
      p.sku,
      p.name AS product_name,
      p.unit_price_cents,
      s.name AS supplier_name,
      s.city AS supplier_city
    FROM products p
    JOIN suppliers s ON s.id = p.supplier_id
    ORDER BY s.id, p.sku
    LIMIT 20
    """
)

## Row counts (sanity check)

Product count stays the same; supplier count equals distinct `supplier_id`s from the 2NF table
(far fewer than products — redundancy removed).

In [ ]:
query(
    """
    SELECT
      (SELECT COUNT(*) FROM products_2nf) AS products_2nf,
      (SELECT COUNT(DISTINCT supplier_id) FROM products_2nf) AS distinct_suppliers_2nf,
      (SELECT COUNT(*) FROM suppliers) AS suppliers,
      (SELECT COUNT(*) FROM products) AS products_3nf
    """
)